# ChemBreak Task Bank Generator V6

This notebook creates harmful chemistry **target goals** for authorized
LLM safety and jailbreak evaluation. It does not generate answers to
those goals.

- **Qwen** and **Ministral** each create one candidate per assignment.
- **Gemma** compares each pair blindly and may reject both.
- `test`: 1 assignment, 2 candidates, 1 pair judgment, up to 1 final goal.
- `pilot`: 9 assignments, 18 candidates, 9 pair judgments, up to 9 final goals.
- `production`: starts with 1,108 assignments and refills deficits until
  exactly 1,000 qualified, balanced, non-duplicate goals are selected.

Only one checkpoint is loaded on the GPU at a time. Every completed row
is saved immediately, so rerunning the main cell resumes the same run.


## 1. Get the current ChemBreak code

The repository is cloned on the first run and fast-forwarded on later runs.
The V6 folder name is explicit so a repository-layout mistake is reported
immediately.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/content/ChemBreak")
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
PROJECT_SUBDIR = "ChemBreak_TaskBank_Generator_v6"

if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. "
        "Rename or remove that Colab folder, then rerun this cell."
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
PIPELINE = PROJECT_DIR / "chembreak_pipeline.py"
REQUIREMENTS = PROJECT_DIR / "requirements_colab.txt"

if not PIPELINE.is_file():
    available = sorted(
        path.name for path in REPO_DIR.iterdir() if path.is_dir()
    )
    raise FileNotFoundError(
        f"Expected {PIPELINE}, but it was not found. "
        f"Top-level repository folders: {available}"
    )

print(f"Ready: {PROJECT_DIR}")
print(f"Python: {sys.executable}")


## 2. Install the small runtime dependency set

Colab's installed PyTorch is kept. This cell adds the loaders needed by
Qwen, Ministral, and Gemma.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)],
    check=True,
)
print("Dependencies installed.")


## 3. Check the GPU, loaders, and public model access

This is a fast check. It does not download the full checkpoint weights
and it does not request a Hugging Face token.


In [ ]:
import os

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PREFLIGHT_DIR = Path("/content/chembreak_v6_preflight")
subprocess.run(
    [
        sys.executable,
        "-u",
        str(PIPELINE),
        "--project-dir",
        str(PROJECT_DIR),
        "--output-dir",
        str(PREFLIGHT_DIR),
        "--run-type",
        "test",
        "--stage",
        "preflight",
    ],
    check=True,
)


## 4. Choose the run

Start with `test`. Change only `RUN_TYPE` when you are ready for the
pilot or production run. Production saves checkpoints to Google Drive.


In [ ]:
RUN_TYPE = "test"  # "test", "pilot", or "production"
PRODUCTION_TARGET = 1000
SESSION_HOURS = 10.5

if RUN_TYPE not in {"test", "pilot", "production"}:
    raise ValueError("RUN_TYPE must be test, pilot, or production.")

if RUN_TYPE == "production":
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(
        "/content/drive/MyDrive/ChemBreak_V6_Results/production"
    )
else:
    OUTPUT_DIR = Path("/content/chembreak_v6_results") / RUN_TYPE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run type: {RUN_TYPE}")
print(f"Output directory: {OUTPUT_DIR}")


## 5. Generate, judge, and select

Progress and errors appear below as they happen. The cell uses unbuffered
output and the same Python interpreter as this notebook. Rerun it to
resume completed candidate and judgment rows.


In [ ]:
command = [
    sys.executable,
    "-u",
    str(PIPELINE),
    "--project-dir",
    str(PROJECT_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--run-type",
    RUN_TYPE,
    "--stage",
    "all",
]
if RUN_TYPE == "production":
    command.extend(
        [
            "--target",
            str(PRODUCTION_TARGET),
            "--session-hours",
            str(SESSION_HOURS),
        ]
    )

print("Starting ChemBreak V6. Progress will appear below.\n", flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)
print("\nChemBreak V6 completed.", flush=True)


## 6. Inspect the task bank and comparison reports

`harmbench_behaviors.csv` is the direct goal list for downstream attack
evaluation. `final_task_bank.csv` keeps the complete ChemBreak metadata.


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / "run_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print("Run summary")
    print(json.dumps(summary, indent=2))
else:
    print("No run summary exists yet.")

for filename in (
    "generator_comparison.csv",
    "coverage_report.csv",
    "final_task_bank.csv",
    "harmbench_behaviors.csv",
):
    path = OUTPUT_DIR / filename
    if path.is_file():
        frame = pd.read_csv(path)
        print(f"\n{filename}: {len(frame):,} row(s)")
        display(frame.head(10))
    else:
        print(f"\n{filename}: not written yet")


## 7. Download the current results

This archives the contents of the selected output directory. It works
for completed runs and for partial resumable runs.


In [ ]:
import shutil
from google.colab import files

result_files = [path for path in OUTPUT_DIR.rglob("*") if path.is_file()]
if not result_files:
    raise FileNotFoundError(f"No result files exist in {OUTPUT_DIR}.")

archive_base = Path("/content") / f"ChemBreak_V6_{RUN_TYPE}_results"
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=str(OUTPUT_DIR),
    )
)
print(f"Created {archive_path} ({archive_path.stat().st_size:,} bytes)")
files.download(str(archive_path))
